In [ ]:
# Notebook 02: Document Processing
# Cell 1
!pip install pymupdf pdfplumber nltk spacy beautifulsoup4 -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:



import json
import re
from pathlib import Path
from typing import List, Dict
import fitz  # PyMuPDF
from tqdm import tqdm
import nltk
import spacy
from collections import defaultdict

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

nlp = spacy.load('en_core_web_sm')

PDF_DIR = BASE_DIR / "data/raw/arxiv_papers"
OUTPUT_DIR = BASE_DIR / "data/processed/chunks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF directory: {PDF_DIR}")
print(f"PDF directory exists: {PDF_DIR.exists()}")

# List files to verify
pdf_files = list(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files")


PDF directory: /content/drive/MyDrive/multi_agent_rag_project/data/raw/arxiv_papers
PDF directory exists: True
Found 527 PDF files


In [ ]:


CONFIG = {
    'chunk_size': 512,
    'chunk_overlap': 128,
    'min_chunk_length': 100,
    'max_chunk_length': 1024,
    'remove_references': True,
    'remove_headers_footers': True,
    'min_words': 20,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Configuration:
  chunk_size: 512
  chunk_overlap: 128
  min_chunk_length: 100
  max_chunk_length: 1024
  remove_references: True
  remove_headers_footers: True
  min_words: 20


In [ ]:



def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using PyMuPDF"""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page_num in range(len(doc)):
            page = doc[page_num]
            text += page.get_text()
        doc.close()
        return text
    except Exception as e:
        print(f"Error processing {pdf_path.name}: {e}")
        return None

def clean_text(text):
    """Clean extracted text"""
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Remove page numbers
    text = re.sub(r'\n\d+\n', '\n', text)
    # Remove excessive newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    return text.strip()

def remove_references_section(text):
    """Remove references section from paper"""
    patterns = [
        r'\n\s*References\s*\n',
        r'\n\s*REFERENCES\s*\n',
        r'\n\s*Bibliography\s*\n',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return text[:match.start()]
    return text

def is_good_chunk(chunk, min_words=20):
    """Filter out low-quality chunks"""
    words = chunk.split()
    if len(words) < min_words:
        return False
    # Check for too many numbers (likely tables)
    num_count = sum(1 for w in words if w.replace('.', '').replace(',', '').isdigit())
    if num_count / len(words) > 0.5:
        return False
    # Check for too many special characters
    special_chars = sum(1 for c in chunk if not c.isalnum() and not c.isspace())
    if special_chars / len(chunk) > 0.3:
        return False
    return True

In [ ]:



def semantic_chunking(text, chunk_size=512, overlap=128):
    """Chunk text based on sentence boundaries"""
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence.split())

        if current_length + sentence_length > chunk_size and current_chunk:
            # Save current chunk
            chunks.append(' '.join(current_chunk))

            # Start new chunk with overlap
            overlap_sentences = []
            overlap_length = 0
            for s in reversed(current_chunk):
                s_len = len(s.split())
                if overlap_length + s_len <= overlap:
                    overlap_sentences.insert(0, s)
                    overlap_length += s_len
                else:
                    break

            current_chunk = overlap_sentences
            current_length = overlap_length

        current_chunk.append(sentence)
        current_length += sentence_length

    # Add remaining chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

In [ ]:


def process_document(pdf_path, paper_metadata, config):
    """Process a single document"""
    # Extract text
    text = extract_text_from_pdf(pdf_path)
    if not text:
        return []

    # Clean text
    text = clean_text(text)

    # Remove references if configured
    if config['remove_references']:
        text = remove_references_section(text)

    # Create chunks
    chunks = semantic_chunking(
        text,
        chunk_size=config['chunk_size'],
        overlap=config['chunk_overlap']
    )

    # Filter and create chunk metadata
    processed_chunks = []
    for i, chunk in enumerate(chunks):
        if not is_good_chunk(chunk, min_words=config['min_words']):
            continue

        chunk_data = {
            'chunk_id': f"{paper_metadata['arxiv_id']}_{i}",
            'text': chunk,
            'arxiv_id': paper_metadata['arxiv_id'],
            'title': paper_metadata['title'],
            'authors': paper_metadata['authors'],
            'abstract': paper_metadata['abstract'],
            'categories': paper_metadata['categories'],
            'published': paper_metadata['published'],
            'chunk_index': i,
            'word_count': len(chunk.split()),
        }
        processed_chunks.append(chunk_data)

    return processed_chunks

In [ ]:


# Load metadata
with open(BASE_DIR /"data/raw/arxiv_metadata.json", 'r') as f:
    papers_metadata = json.load(f)

# Create metadata lookup
metadata_lookup = {p['arxiv_id']: p for p in papers_metadata}

print(f"Loaded metadata for {len(papers_metadata)} papers")

Loaded metadata for 527 papers


In [ ]:



# Save chunks
chunks_file = CHUNKS_DIR / "all_chunks.json"
with open(chunks_file, 'w') as f:
    json.dump(all_chunks, f, indent=2)

stats_file = OUTPUT_DIR / "processing_stats.json"
with open(stats_file, 'w') as f:
    json.dump(processing_stats, f, indent=2)

print(f"\nSaved {len(all_chunks)} chunks to {chunks_file}")
print(f"Saved stats to {stats_file}")


# Analyze chunks
word_counts = [c['word_count'] for c in all_chunks]
print(f"\nChunk Statistics:")
print(f"  Min words: {min(word_counts)}")
print(f"  Max words: {max(word_counts)}")
print(f"  Mean words: {sum(word_counts)/len(word_counts):.1f}")
print(f"  Total words: {sum(word_counts):,}")

print("\nNotebook 02 Complete!")


Saved 12728 chunks to /content/drive/MyDrive/multi_agent_rag_project/data/processed/chunks/all_chunks.json
Saved stats to /content/drive/MyDrive/multi_agent_rag_project/data/processed/chunks/processing_stats.json

Chunk Statistics:
  Min words: 27
  Max words: 1256
  Mean words: 490.0
  Total words: 6,236,234

Notebook 02 Complete!
